In [2]:
import numpy as np
from collections import defaultdict

class SimpleGridworld:
    def __init__(self):
        self.height = 4
        self.width = 4
        self.states = list(range(16))
        self.terminal_states = {0, 15}

        self.actions = [0, 1, 2, 3]
        self.action_effects = {
            0: (-1, 0),
            1: (0, 1),
            2: (1, 0),
            3: (0, -1)
        }

    def reset(self):
        self.state = np.random.choice([s for s in self.states if s not in self.terminal_states])
        return self.state

    def step(self, action):
        if self.state in self.terminal_states:
            return self.state, 0, True

        row, col = self.state // self.width, self.state % self.width
        dr, dc = self.action_effects[action]

        new_row = max(0, min(self.height - 1, row + dr))
        new_col = max(0, min(self.width - 1, col + dc))

        self.state = new_row * self.width + new_col

        reward = -1
        done = self.state in self.terminal_states

        return self.state, reward, done

def sample_policy(state):
    return np.random.choice([0, 1, 2, 3])

def generate_episode(env, policy):
    episode = []
    state = env.reset()
    done = False

    while not done:
        action = policy(state)
        next_state, reward, done = env.step(action)
        episode.append((state, action, reward))
        state = next_state

    return episode

def monte_carlo_prediction(env, policy, num_episodes, gamma=1.0):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(float)
    V = np.zeros(len(env.states))

    for _ in range(num_episodes):
        episode = generate_episode(env, policy)
        states_in_episode = [step[0] for step in episode]

        G = 0
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward

            if state not in states_in_episode[:t]:
                returns_sum[state] += G
                returns_count[state] += 1
                V[state] = returns_sum[state] / returns_count[state]

    return V

if __name__ == "__main__":
    env = SimpleGridworld()
    num_episodes = 5000

    print(f"Running First-Visit MC Prediction for {num_episodes} episodes...")
    state_values = monte_carlo_prediction(env, sample_policy, num_episodes=num_episodes)

    print("\nEstimated State-Value Function V(s):")
    print(state_values.reshape(4, 4))


Running First-Visit MC Prediction for 5000 episodes...

Estimated State-Value Function V(s):
[[  0.         -14.2536036  -20.85202639 -22.82232613]
 [-13.84033246 -18.4255397  -20.5915493  -20.8012218 ]
 [-20.14130926 -20.49625749 -18.5493782  -14.8896427 ]
 [-22.27455357 -20.43454545 -14.33701413   0.        ]]
